# Benchmark dataloders of Tahoe100M dataset

In [1]:
from pathlib import Path

import json

In [2]:
PATH_ZARR = Path("/vol/data/annbatch_benchmark/tahoe100M/zarr_shuffled")
PATH_H5AD = Path("/vol/data/annbatch_benchmark/tahoe100M/h5ad_shuffled")

## Arrayloaders

In [3]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=64 --preload_nchunks=256 --n_samples=2000000 --store_path={PATH_ZARR}

In [4]:
arrayloaders_64 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_64

36105.55313837123

In [5]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=128 --preload_nchunks=128 --n_samples=2000000 --store_path={PATH_ZARR}

In [6]:
arrayloaders_128 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_128

44169.93670323333

In [7]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=256 --preload_nchunks=64 --n_samples=2000000 --store_path={PATH_ZARR}

In [8]:
arrayloaders_256 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_256

51618.77520406477

In [9]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=512 --preload_nchunks=32 --n_samples=2000000 --store_path={PATH_ZARR}

In [10]:
arrayloaders_512 = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_512

54469.62571534467

In [11]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/annbatch/benchmark_annbatch.py --chunk_size=512 --preload_nchunks=32 --preload_to_gpu=True --n_samples=2000000 --store_path={PATH_ZARR}

In [12]:
arrayloaders_gpu = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
arrayloaders_gpu


83578.99769788743

## MappedCollection

In [13]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/mapped_collection/benchmark_mapped_collection.py --store_path={PATH_H5AD}

In [14]:
mapped_collection = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
mapped_collection

1107.9764873478719

## scDataset

In [15]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/scDataset/benchmark_scDataset.py --num_workers=8 --batch_size=64 --store_path={PATH_H5AD}  # recommended settings from tutorial

In [16]:
sc_dataset = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset

1646.8910246620324

In [20]:
%%capture cap --no-stderr
!python arrayloader_benchmarks/scDataset/benchmark_scDataset.py --num_workers=8 --batch_size=4096 --block_size=256 --fetch_factor=4 --store_path={PATH_H5AD}

In [21]:
sc_dataset_matched = json.loads(cap.stdout.split("\n")[-2].rstrip())["samples/sec"]  # noqa: F821
sc_dataset_matched

19619.49772167297

## Plot results

In [22]:
import pandas as pd

results = pd.DataFrame(
    {
        "loader": [
            "annbatch (chunk_size=64)",
            "annbatch (chunk_size=128)",
            "annbatch (chunk_size=256)",
            "annbatch (chunk_size=512)",
            "annbatch (GPU)",
            "MappedCollection",
            "scDataset (recommended settings)",
            "scDataset (annbatch matched settings)",
        ],
        "samples/sec": [
            arrayloaders_64,
            arrayloaders_128,
            arrayloaders_256,
            arrayloaders_512,
            arrayloaders_gpu,
            mapped_collection,
            sc_dataset,
            sc_dataset_matched,
        ],
    }
)
results.to_csv("loading_speed_results.csv", index=False)
results

,loader,samples/sec
0,annbatch (chunk_size=64),36105.553138
1,annbatch (chunk_size=128),44169.936703
2,annbatch (chunk_size=256),51618.775204
3,annbatch (chunk_size=512),54469.625715
4,annbatch (GPU),83578.997698
5,MappedCollection,1107.976487
6,scDataset (recommended settings),1646.891025
7,scDataset (annbatch matched settings),19619.497722
